## PEFT库QLoRA实战-ChatGLM3-6B

In [1]:
import torch
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments, BitsAndBytesConfig
from datasets import load_dataset
from peft import TaskType, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft.utils import TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING
from typing import List, Dict

def train_chatglm_qlora(training_args: TrainingArguments):
    # 全局变量和参数设置
    model_name_or_path = '/root/dataDisk/hf/hub/models/chatglm3-6b'
    train_data_path = '/root/dataDisk/hf/hub/datasets/adgen'
    seed = 8
    max_input_length = 512
    max_output_length = 1536
    lora_rank = 4
    lora_alpha = 32
    lora_dropout = 0.05
    prompt_text = ''
    compute_dtype = 'fp32'

    # 加载数据集
    dataset = load_dataset(train_data_path)
    
    # 加载tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True, revision='b098244')

    # 定义tokenize函数
    def tokenize_func(example, tokenizer, ignore_label_id=-100):
        question = prompt_text + example['content']
        if example.get('input', None) and example['input'].strip():
            question += f'\n{example["input"]}'
        answer = example['summary']
        q_ids = tokenizer.encode(text=question, add_special_tokens=False)
        a_ids = tokenizer.encode(text=answer, add_special_tokens=False)
        if len(q_ids) > max_input_length - 2:
            q_ids = q_ids[:max_input_length - 2]
        if len(a_ids) > max_output_length - 1:
            a_ids = a_ids[:max_output_length - 1]
        input_ids = tokenizer.build_inputs_with_special_tokens(q_ids, a_ids)
        question_length = len(q_ids) + 2
        labels = [ignore_label_id] * question_length + input_ids[question_length:]
        return {'input_ids': input_ids, 'labels': labels}

    # 处理数据集
    column_names = dataset['train'].column_names
    tokenized_dataset = dataset['train'].map(
        lambda example: tokenize_func(example, tokenizer),
        batched=False, 
        remove_columns=column_names
    )
    tokenized_dataset = tokenized_dataset.shuffle(seed=seed).flatten_indices()

    # 定义DataCollator
    class DataCollatorForChatGLM:
        def __init__(self, pad_token_id: int, max_length: int = 2048, ignore_label_id: int = -100):
            self.pad_token_id = pad_token_id
            self.ignore_label_id = ignore_label_id
            self.max_length = max_length

        def __call__(self, batch_data: List[Dict[str, List]]) -> Dict[str, torch.Tensor]:
            len_list = [len(d['input_ids']) for d in batch_data]
            batch_max_len = max(len_list)
            input_ids, labels = [], []
            for len_of_d, d in sorted(zip(len_list, batch_data), key=lambda x: -x[0]):
                pad_len = batch_max_len - len_of_d
                ids = d['input_ids'] + [self.pad_token_id] * pad_len
                label = d['labels'] + [self.ignore_label_id] * pad_len
                if batch_max_len > self.max_length:
                    ids = ids[:self.max_length]
                    label = label[:self.max_length]
                input_ids.append(torch.LongTensor(ids))
                labels.append(torch.LongTensor(label))
            input_ids = torch.stack(input_ids)
            labels = torch.stack(labels)
            return {'input_ids': input_ids, 'labels': labels}

    data_collator = DataCollatorForChatGLM(pad_token_id=tokenizer.pad_token_id)

    # 加载模型
    _compute_dtype_map = {'fp32': torch.float32, 'fp16': torch.float16, 'bf16': torch.bfloat16}
    
    q_config = BitsAndBytesConfig(load_in_4bit=True, 
                                  bnb_4bit_quant_type='nf4', 
                                  bnb_4bit_use_double_quant=True, 
                                  bnb_4bit_compute_dtype=_compute_dtype_map[compute_dtype])
    
    model = AutoModel.from_pretrained(model_name_or_path, 
                                      quantization_config=q_config, 
                                      device_map='auto', 
                                      trust_remote_code=True, 
                                      revision='b098244')

    # 预处理量化模型
    kbit_model = prepare_model_for_kbit_training(model)

    # LoRA配置
    target_modules = TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING['chatglm']
    lora_config = LoraConfig(
        target_modules=target_modules,
        r=lora_rank,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias='none',
        inference_mode=False,
        task_type=TaskType.CAUSAL_LM
    )
    qlora_model = get_peft_model(kbit_model, lora_config)

    # 创建Trainer并开始训练
    trainer = Trainer(
        model=qlora_model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator
    )
    
    trainer.train()
    
    # 保存模型
    trainer.model.save_pretrained(training_args.output_dir)

/root/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 使用示例
demo_args = TrainingArguments(
    output_dir="models/chatglm3-6b_finetuned",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-3,
    max_steps=400,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="steps",
    save_steps=20,
    optim="adamw_torch",
    fp16=True,
)

In [3]:
train_chatglm_qlora(demo_args)

Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards: 100%|██████████| 7/7 [00:03<00:00,  2.13it/s]
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/root/.local/lib/python3.10/site-packages/bitsandbytes/nn/modules.py:426: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


Step,Training Loss
10,5.017400
20,4.221500
30,3.815000
40,3.684200
50,3.585800
60,3.522700
70,3.565700
80,3.493400
90,3.449300
100,3.536700


Checkpoint destination directory models/demo/chatglm3-6b/final_model/checkpoint-20 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory models/demo/chatglm3-6b/final_model/checkpoint-40 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory models/demo/chatglm3-6b/final_model/checkpoint-60 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory models/demo/chatglm3-6b/final_model/checkpoint-80 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory models/demo/chatglm3-6b/final_model/checkpoint-100 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory models/demo/chatglm3-6b/final_model/checkpoint-120 already exists and is non-empty.Saving will proceed but saved results 

In [2]:
# 设置训练参数
training_args = TrainingArguments(
    output_dir="models/chatglm3-6b_finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8, # 增加以减少显存消耗
    max_steps=625,  # 确保至少处理10,000个样本
    learning_rate=1e-4,  # 可以根据实际情况调整学习率
    lr_scheduler_type="linear",
    warmup_steps=50,  # 根据总步数的一定比例设置预热步数
    logging_steps=10,  # 每10步记录一次日志
    save_steps=200,  # 每200步保存一次模型
    save_total_limit=3,  # 限制保存的总模型数量
    evaluation_strategy="no",  # 如果有验证集，可以设置为"steps"或"epoch"
    report_to="none",  # 如果不需要Hugging Face Hub的集成，设置为"none"
    load_best_model_at_end=False,  # 如果使用验证，可以考虑加载最佳模型
    metric_for_best_model="loss",  # 如果使用验证，设置基于哪个指标选择最佳模型
    optim="adamw_torch",  # 使用 AdamW 优化器
    fp16=True  # 使用混合精度训练
)

# 然后调用训练函数
train_chatglm_qlora(training_args)

Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards: 100%|██████████| 7/7 [00:03<00:00,  2.23it/s]
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/root/.local/lib/python3.10/site-packages/bitsandbytes/nn/modules.py:426: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


Step,Training Loss
10,5.143700
20,5.124800
30,4.847700
40,4.405300
50,4.077400
60,3.879400
70,3.840200
80,3.726600
90,3.645900
100,3.690800


Checkpoint destination directory models/chatglm3-6b_finetuned/checkpoint-200 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory models/chatglm3-6b_finetuned/checkpoint-400 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory models/chatglm3-6b_finetuned/checkpoint-600 already exists and is non-empty.Saving will proceed but saved results may be invalid.


## 模型推理-使用QLoRA微调后的ChatGLM3-6B

In [3]:
import torch
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig

# 定义全局变量和参数
model_name_or_path = '/root/dataDisk/hf/hub/models/chatglm3-6b'  # 模型ID或本地路径
peft_model_path = "models/chatglm3-6b_finetuned"

# 加载PEFT配置
config = PeftConfig.from_pretrained(peft_model_path)

# 设置量化配置
q_config = BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=torch.float32)

# 加载基础模型
base_model = AutoModel.from_pretrained(model_name_or_path,
                                       quantization_config=q_config,
                                       trust_remote_code=True,
                                       device_map='auto')
base_model.requires_grad_(False)
base_model.eval()

# 加载Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True)

# 输入文本
input_text = '类型#裙*版型#显瘦*风格#文艺*风格#简约*图案#印花*图案#撞色*裙下摆#压褶*裙长#连衣裙*裙领型#圆领'
print(f'输入：\n{input_text}')

# 微调前的输出
response_pre, history_pre = base_model.chat(tokenizer=tokenizer, query=input_text)
print(f'ChatGLM3-6B 微调前：\n{response_pre}')

# 加载微调后的模型
peft_model = PeftModel.from_pretrained(base_model, peft_model_path)

# 微调后的输出
response_post, history_post = peft_model.chat(tokenizer=tokenizer, query=input_text)
print(f'ChatGLM3-6B 微调后: \n{response_post}')

Loading checkpoint shards: 100%|██████████| 7/7 [00:03<00:00,  2.18it/s]
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.


输入：
类型#裙*版型#显瘦*风格#文艺*风格#简约*图案#印花*图案#撞色*裙下摆#压褶*裙长#连衣裙*裙领型#圆领
ChatGLM3-6B 微调前：
这款连衣裙以其文艺风格和简约图案受到了许多消费者的喜爱。它的下摆采用了印花图案和撞色设计，使得整个裙子更加生动有趣。压褶的设计使得裙长更加流畅，而裙领型的圆领则增添了一丝柔美。这款裙子在显瘦方面也表现出色，让人在穿上它后瞬间变得优雅起来。
ChatGLM3-6B 微调后: 
这款连衣裙的圆领设计，能够修饰颈部线条，显脸型，又显得文艺优雅。同时，胸前的小印花，与撞色的<UNK>，让连衣裙充满了时尚气息。而裙身压褶的点缀，增添女性柔美的气质。
